### USE GPU RUNTIME

In [ ]:
!pip install -U transformers==4.56.1 datasets==4.0.0 evaluate==0.4.5 accelerate==1.10.1 torch==2.6.0 pandas==2.2.2 numpy==2.0.2 scipy==1.16.3

In [ ]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import evaluate, numpy as np, torch
import warnings
warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

dataset = load_dataset("glue", "sst2")

model_name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)
def tok_fn(b): return tok(b["sentence"], truncation=True)
data = dataset.map(tok_fn, batched=True, remove_columns=["sentence","idx"])

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

# Freeze base initially (feature extraction); train only classification head
for p in model.base_model.parameters():
    p.requires_grad = False

acc = evaluate.load("accuracy")
def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": acc.compute(predictions=preds, references=labels)["accuracy"]}

args = TrainingArguments(
    output_dir="sst2-distilbert",
    learning_rate=5e-4,                     # higher LR since head-only
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,                     # for quick demo. bump to 2–3 for better
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=42,
)

train_ds = data["train"].shuffle(seed=42).select(range(8000))

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=data["validation"],
    processing_class=tok,
    data_collator=DataCollatorWithPadding(tokenizer=tok),
    compute_metrics=metrics,
)

print("WARM-UP (head only):")
trainer.train()

# Unfreeze top transformer layers
for p in model.base_model.parameters():
    p.requires_grad = False
for name, param in model.base_model.named_parameters():
    if any(k in name for k in ["transformer.layer.4", "transformer.layer.5"]):
        param.requires_grad = True

# Lower LR for gentle fine-tune
fine_args = TrainingArguments(
    **{**args.to_dict(), "learning_rate": 2e-5, "num_train_epochs": 1}
)

# Rebuild Trainer
trainer = Trainer(
    model=model,
    args=fine_args,
    train_dataset=train_ds,
    eval_dataset=data["validation"],
    processing_class=tok,
    data_collator=DataCollatorWithPadding(tokenizer=tok),
    compute_metrics=metrics,
)

print("FINE-TUNE (top layers + head):")
trainer.train()

print("Eval:", trainer.evaluate())